# Daraz review sentiment training

This notebook runs the complete project in Google Colab. It uses `daraz_review_filtered2.csv`, the `Review` text column, and the `sentiment` label column.

Run the cells from top to bottom. When prompted, upload `data/daraz_review_filtered2.csv`.

In [ ]:
!pip -q install transformers datasets accelerate seaborn scikit-learn pandas matplotlib

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC


## Upload and prepare the data

In [ ]:
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_names:
    raise FileNotFoundError('Upload a CSV file, preferably daraz_review_filtered2.csv.')

csv_path = Path(csv_names[0])
raw_data = pd.read_csv(csv_path)
print('Input columns:', list(raw_data.columns))
print('Input rows:', len(raw_data))

TEXT_COLUMN = 'Review'
LABEL_COLUMN = 'sentiment'
LABEL_ORDER = ['negative', 'neutral', 'positive']

for required_column in [TEXT_COLUMN, LABEL_COLUMN]:
    if required_column not in raw_data.columns:
        raise ValueError(f"Missing required column: {required_column}")

def clean_text(value):
    if pd.isna(value):
        return ''
    text = str(value)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

labels = raw_data[LABEL_COLUMN].astype('string').str.strip().str.lower()
if (~labels.isin(LABEL_ORDER)).any():
    raise ValueError('Labels must be negative, neutral, or positive.')

data = pd.DataFrame({
    'text': raw_data[TEXT_COLUMN].fillna('').map(clean_text),
    'label': labels,
})
data = data[data['text'] != '']
data = data.drop_duplicates(subset=['text']).reset_index(drop=True)

print('Rows used:', len(data))
display(data.head())
display(data['label'].value_counts().reindex(LABEL_ORDER, fill_value=0))

In [ ]:
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['label'],
)
print(f'Training rows: {len(train_data)}')
print(f'Test rows: {len(test_data)}')

## Train and evaluate classical models

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
train_features = vectorizer.fit_transform(train_data['text'])
test_features = vectorizer.transform(test_data['text'])

def metric_summary(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=LABEL_ORDER, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(actual, predicted)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'classification_report': classification_report(
            actual, predicted, labels=LABEL_ORDER, zero_division=0, output_dict=True
        ),
    }

def show_confusion_matrix(actual, predicted, title):
    matrix = confusion_matrix(actual, predicted, labels=LABEL_ORDER)
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER)
    plt.xlabel('Predicted label')
    plt.ylabel('Actual label')
    plt.title(title)
    plt.show()

classical_results = {}
models = {
    'tfidf_logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'tfidf_linear_svm': LinearSVC(class_weight='balanced'),
}

for name, model in models.items():
    model.fit(train_features, train_data['label'])
    predictions = model.predict(test_features)
    classical_results[name] = metric_summary(test_data['label'], predictions)
    print(name, classical_results[name]['accuracy'], classical_results[name]['macro_f1'])
    show_confusion_matrix(test_data['label'], predictions, name)


## Train and evaluate XLM-R

Enable a Colab GPU for faster training: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

MODEL_NAME = 'xlm-roberta-base'
label_to_number = {'negative': 0, 'neutral': 1, 'positive': 2}
number_to_label = {value: key for key, value in label_to_number.items()}
train_for_model = train_data.copy()
test_for_model = test_data.copy()
train_for_model['label'] = train_for_model['label'].map(label_to_number)
test_for_model['label'] = test_for_model['label'].map(label_to_number)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label_to_number,
    id2label=number_to_label,
)

train_dataset = Dataset.from_pandas(train_for_model, preserve_index=False)
test_dataset = Dataset.from_pandas(test_for_model, preserve_index=False)

def tokenize_reviews(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_reviews, batched=True)
test_dataset = test_dataset.map(tokenize_reviews, batched=True)

def calculate_metrics(result):
    predictions = np.argmax(result.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        result.label_ids, predictions, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(result.label_ids, predictions)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
    }

training_args = TrainingArguments(
    output_dir='colab_temp_training_output',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=calculate_metrics,
)
trainer.train()
transformer_results = trainer.evaluate()
print(transformer_results)

In [ ]:
prediction_result = trainer.predict(test_dataset)
predicted_numbers = np.argmax(prediction_result.predictions, axis=1)
predicted_labels = pd.Series(predicted_numbers).map(number_to_label)
show_confusion_matrix(test_data['label'], predicted_labels, 'xlm_roberta')

all_results = {
    'dataset': {
        'file': csv_path.name,
        'text_column': TEXT_COLUMN,
        'label_column': LABEL_COLUMN,
        'rows_used': len(data),
        'class_distribution': data['label'].value_counts().reindex(LABEL_ORDER, fill_value=0).to_dict(),
    },
    'classical_models': classical_results,
    'transformer': {key: float(value) for key, value in transformer_results.items() if isinstance(value, (int, float))},
}
display(pd.DataFrame({
    'model': ['Logistic Regression', 'Linear SVM', 'XLM-R'],
    'accuracy': [classical_results['tfidf_logistic_regression']['accuracy'], classical_results['tfidf_linear_svm']['accuracy'], all_results['transformer'].get('eval_accuracy')],
    'macro_f1': [classical_results['tfidf_logistic_regression']['macro_f1'], classical_results['tfidf_linear_svm']['macro_f1'], all_results['transformer'].get('eval_macro_f1')],
}))


## Test a new review

In [ ]:
new_review = 'good quality product and fast delivery'
new_features = vectorizer.transform([clean_text(new_review)])
print('Logistic Regression:', models['tfidf_logistic_regression'].predict(new_features)[0])
print('Linear SVM:', models['tfidf_linear_svm'].predict(new_features)[0])

encoded_review = tokenizer(new_review, return_tensors='pt', truncation=True, max_length=256)
encoded_review = {key: value.to(model.device) for key, value in encoded_review.items()}
with torch.no_grad():
    logits = model(**encoded_review).logits
predicted_number = torch.argmax(logits, dim=1).item()
confidence = torch.softmax(logits, dim=1)[0][predicted_number].item()
print('XLM-R:', number_to_label[predicted_number])
print(f'XLM-R confidence: {confidence:.2%}')